# NB04 — Word2Vec Embeddings  |  Claude Assisted

**NLP Workshop Series — Quran Translations Dataset**  
**Notebook 4 of 8**

---

## Section 0: Introduction and Workshop Roadmap

### Full 8-Notebook Series

```
+-------+--------------------------------------+------------+
| NB    | Topic                                | Status     |
+-------+--------------------------------------+------------+
| NB01  | Text Preprocessing & Bag of Words    | Done       |
| NB02  | TF-IDF & Ranked Retrieval            | Done       |
| NB03  | N-Grams & Context Windows            | Done       |
| NB04  | Word2Vec Embeddings                  | <== HERE   |
| NB05  | FastText & Subword Embeddings        | Next       |
| NB06  | Sentence Embeddings (SBERT)          | Upcoming   |
| NB07  | Fine-Tuning on Quran Corpus          | Upcoming   |
| NB08  | Capstone: Full Semantic Search App   | Upcoming   |
+-------+--------------------------------------+------------+
```

---

### Recap: What Classical Methods Couldn't Do

In the first three notebooks we built increasingly sophisticated text representations:

- **NB01 — Bag of Words**: Treated each word as an independent dimension. "Mercy" and "compassion" live in completely different columns with no relationship at all. Searching for "kindness" returns zero results if the word does not appear verbatim in any verse.

- **NB02 — TF-IDF**: Improved by down-weighting common words and up-weighting rare ones. But still fundamentally a bag of words: vectors are sparse, high-dimensional, and have no notion of semantic similarity between words.

- **NB03 — N-Grams**: Captured local word order and co-occurrence statistics at the surface level. Better for language modeling and phrase detection, but still not dense representations. "Mercy" and "compassion" remain unrelated.

**The shared failure**: All classical methods are *counting methods*. They count word occurrences and produce *sparse, high-dimensional vectors*. In a vocabulary of 5,000 words, a verse vector has 5,000 dimensions and almost all values are zero. There is no geometry that places similar concepts near each other.

---

### The Big Leap: From Sparse to Dense, From Counting to Meaning

Word2Vec (Mikolov et al., 2013, Google) was a revolution. Instead of counting, a neural network *learns* word representations from context. The result:

- **Dense vectors**: 100 or 300 real-valued numbers instead of 5,000 zeros
- **Semantic geometry**: Similar words are *close* in vector space
- **Analogical reasoning**: king - man + woman = queen
- **Generalisation**: You can find "mercy" verses by searching "kindness" — because the model knows they are related

This is the backbone of every modern NLP system, from BERT to GPT.

---

### Learning Objectives

By the end of this notebook you will be able to:

1. Explain the distributional hypothesis and why it enables word embeddings
2. Describe CBOW and Skip-Gram architectures
3. Train a Word2Vec model on a real Arabic-to-English translation corpus
4. Explore word similarity, analogies, and semantic clustering
5. Visualise embeddings in 2D using PCA and t-SNE
6. Build a semantic search engine that solves the "kindness" problem from NB01
7. Articulate the limitations of Word2Vec and why we need FastText next

## Section 1: Why Embeddings?

### The Core Problem with Sparse Representations

Imagine a vocabulary of 5,000 unique words. In a Bag-of-Words model, each word is assigned an index:

```
Word         Index
--------     -----
...
compassion   891
...
mercy        4521
...
```

The vector for the word "mercy" looks like this:

```
[0, 0, 0, ..., 0, 1, 0, ..., 0]
                   ^-- position 4521 only
```

The vector for "compassion":

```
[0, 0, 0, ..., 0, 1, 0, ..., 0]
               ^-- position 891 only
```

These two vectors are *orthogonal*. Their dot product is zero. Their cosine similarity is zero. According to BoW geometry, "mercy" and "compassion" are as unrelated as "mercy" and "elephant".

---

### Dense vs Sparse: ASCII Diagram

```
SPARSE (BoW / TF-IDF)
  mercy:      [0, 0, 0, 0, 0, 1, 0, 0, 0, ..., 0]   -- 5000 dims
  compassion: [0, 0, 1, 0, 0, 0, 0, 0, 0, ..., 0]   -- 5000 dims
  Cosine similarity = 0.0  (completely unrelated!)

DENSE (Word2Vec)
  mercy:      [0.32, -0.14,  0.87, ...,  0.45]   -- 100 dims
  compassion: [0.29, -0.18,  0.81, ...,  0.51]   -- 100 dims
  Cosine similarity = 0.89  (very close!)
```

The dense representation captures *meaning* because the network learned that "mercy" and "compassion" appear in similar contexts throughout the corpus.

---

### The Distributional Hypothesis

> *"You shall know a word by the company it keeps."*  
> — J.R. Firth, 1957

This is the foundational insight behind all word embeddings:

- Words that appear in similar *contexts* tend to have similar *meanings*
- "mercy" appears near: forgiveness, compassion, kindness, lord, gracious
- "punishment" appears near: fire, hell, torment, wicked, unbelievers
- A neural network trained to predict context words learns to give similar vectors to words with similar contexts

Word2Vec operationalises this 1957 linguistic insight at scale using neural networks.

In [1]:
# Section 1 — Demo: BoW vector sparsity vs what we want
import numpy as np

# Simulate a small vocabulary
vocab = ['allah', 'mercy', 'fire', 'prayer', 'compassion', 'punishment',
         'believers', 'faith', 'hell', 'righteous', 'forgiving', 'gracious']
word2idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

# BoW one-hot vectors
def bow_vector(word, word2idx, V):
    vec = np.zeros(V)
    if word in word2idx:
        vec[word2idx[word]] = 1
    return vec

mercy_bow = bow_vector('mercy', word2idx, V)
compassion_bow = bow_vector('compassion', word2idx, V)

cos_sim = np.dot(mercy_bow, compassion_bow) / (
    (np.linalg.norm(mercy_bow) * np.linalg.norm(compassion_bow)) + 1e-10
)

print("=== BoW Sparse Vectors (vocabulary size:", V, ")===")
print("mercy      :", mercy_bow)
print("compassion :", compassion_bow)
print("Cosine similarity (mercy vs compassion):", cos_sim)
print()
print("Observation: cos_sim = 0.0 — BoW treats them as completely unrelated!")
print()
print("In a real 5000-word vocabulary, each vector has 4999 zeros.")
print("Word2Vec will learn 100-dimensional DENSE vectors where")
print("mercy and compassion are close (sim > 0.8) because they")
print("appear in similar Quranic contexts.")

=== BoW Sparse Vectors (vocabulary size: 12 )===
mercy      : [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
compassion : [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
Cosine similarity (mercy vs compassion): 0.0

Observation: cos_sim = 0.0 — BoW treats them as completely unrelated!

In a real 5000-word vocabulary, each vector has 4999 zeros.
Word2Vec will learn 100-dimensional DENSE vectors where
mercy and compassion are close (sim > 0.8) because they
appear in similar Quranic contexts.


## Section 2: Word2Vec Theory (Detailed)

### What Is Word2Vec?

Word2Vec is a shallow neural network trained to predict words from context (or context from words). The *weights* of that network — not the predictions — become the word embeddings. The network is a means to an end: we throw away the predictions and keep the learned representations.

Published by Mikolov et al. at Google in 2013, it remains one of the most influential papers in NLP history.

---

### Architecture 1: CBOW (Continuous Bag of Words)

Given a *context window* of surrounding words, predict the *center word*.

```
Context words (input):                 Center word (output):

  [the]  [most]  ___  [lord]  [is]  -->  [merciful]

Neural Network:

  the    --->
  most   ---->  [Average]  -->  [Hidden: 100 dims]  -->  [Softmax over vocab]
  lord   ---->
  is     --->

Training signal: did we predict "merciful" correctly?
```

CBOW is **faster to train** and works well for frequent words.

---

### Architecture 2: Skip-Gram

Given the *center word*, predict the surrounding *context words*. This is the inverse of CBOW.

```
Center word (input):        Context words (output):

  [merciful]   -->   predict: [the] [most] [lord] [is]

Neural Network:

  merciful  -->  [Embedding: 100 dims]  -->  [Softmax over vocab]
                                              for EACH context position

Training signal: did we predict each context word correctly?
```

Skip-Gram is **slower to train** but produces **better representations for rare words** — ideal for domain-specific corpora like the Quran where many theological terms are infrequent.

We use **sg=1** (Skip-Gram) in this notebook.

---

### Training Objective

For Skip-Gram, the objective is to maximise the probability of observing the actual context words given the center word:

```
Maximise:  sum over all (center, context) pairs of  log P(context | center)

Where:     P(context | center)  is computed via softmax over the entire vocabulary
```

In practice, **Negative Sampling** is used: instead of computing softmax over all 5,000+ words (expensive), the model contrasts the real context word against a few randomly sampled "noise" words. This makes training feasible.

---

### Why 100-300 Dimensions?

This is empirically determined:

- **Too few (< 50)**: The model cannot fit enough information — related words may not cluster cleanly
- **100-300**: Sweet spot — captures rich semantics without overfitting on small corpora
- **Too many (> 500)**: Diminishing returns, slower training, more memory, risk of overfitting

For a corpus of ~6,000 verses (small), **100 dimensions** is appropriate.

---

### The Famous Analogy: king - man + woman = queen

```
vector("king")   - vector("man")   + vector("woman")  ~=  vector("queen")

Geometric interpretation:
  The direction from "man" to "king" encodes "royalty".
  Apply that same direction to "woman" and you arrive near "queen".

This works because:
  king and man share a "male" component.
  Subtracting that leaves a "royalty" direction.
  Adding it to "woman" (female royalty) lands near "queen".
```

This emergent property — arithmetic in meaning space — is what makes Word2Vec so powerful and surprising.

---

### Context Window Size

The window parameter controls how many words on each side count as context:

```
window=2 around "merciful" in: "He is the most merciful lord of all"

  Context: [the, most, lord, of]

Small window  --> captures syntactic relationships (grammatical)
Large window  --> captures semantic/topical relationships (meaning)

window=5 is standard for semantic tasks.
```

## Section 3: Training Word2Vec on the Quran Corpus

In [2]:
# Section 3 — Load dataset with robust path finder
import os
import pandas as pd

paths_to_try = [
    '../quran_translations.csv',
    './quran_translations.csv',
    '../../quran_translations.csv',
    os.path.join(os.path.dirname(os.getcwd()), 'quran_translations.csv')
]

df = None
for p in paths_to_try:
    if os.path.exists(p):
        df = pd.read_csv(p)
        print("Loaded from:", p)
        break

if df is None:
    raise FileNotFoundError("quran_translations.csv not found. Tried: " + str(paths_to_try))

print("Shape:", df.shape)
print("Columns:", list(df.columns)[:10])
print()
print(df['daryabadi'].head(3).to_string())

Loaded from: ../quran_translations.csv
Shape: (6236, 18)
Columns: ['Surah', 'Verse', 'ahmedali', 'ahmedraza', 'arberry', 'daryabadi', 'hilali', 'itani', 'maududi', 'mubarakpuri']

0    In the name of Allah, the Compassionate, the M...
1    All praise unto Allah, the Lord of all the wor...
2                     The Compassionate, the Merciful.


In [3]:
# Section 3 — Tokenize corpus for Word2Vec
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))

def tokenize_verse(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Build list of token lists (one per verse)
corpus_sentences = [tokenize_verse(v) for v in df['daryabadi'].dropna()]

# Filter out empty verses
corpus_sentences = [s for s in corpus_sentences if len(s) >= 3]

print("Total verses (tokenized):", len(corpus_sentences))
print()
print("Sample verse tokens:")
for i in [0, 100, 500]:
    print(" Verse", i, ":", corpus_sentences[i][:15], '...')

Total verses (tokenized): 5962

Sample verse tokens:
 Verse 0 : ['name', 'allah', 'compassionate', 'merciful'] ...
 Verse 100 : ['means', 'ever', 'wish', 'hands', 'sent', 'allah', 'knower', 'wrong', 'doers'] ...
 Verse 500 : ['let', 'beware', 'leave', 'behind', 'weakly', 'progeny', 'would', 'afraid', 'account', 'let', 'wherefore', 'fear', 'allah', 'says', 'proper'] ...


In [4]:
# Section 3 - Train Word2Vec: Skip-Gram AND CBOW
# We use a shared hyperparameter dictionary for both models
# This ensures a fair comparison and teaches good reproducibility practice

from gensim.models import Word2Vec

# Shared hyperparameter config -- change here, affects both models
W2V_KW = dict(
    vector_size = 100,   # embedding dimensions
    window      = 5,     # context window size (words on each side)
    min_count   = 2,     # ignore words appearing fewer than 2 times
    workers     = 4,     # parallel threads
    epochs      = 50,    # training passes over corpus
    seed        = 42,    # reproducibility
)

# Train Skip-Gram model (sg=1)
# Skip-Gram: predicts surrounding context words from the center word
# Better for rare words and larger datasets
print('Training Skip-Gram model...')
sg_model = Word2Vec(sentences=tokenized_corpus, sg=1, **W2V_KW)
print('Skip-Gram training complete.')
print('Vocabulary size:', len(sg_model.wv))

# Train CBOW model (sg=0)
# CBOW: predicts the center word from surrounding context words
# Faster to train, tends to produce tighter clusters
print()
print('Training CBOW model...')
cbow_model = Word2Vec(sentences=tokenized_corpus, sg=0, **W2V_KW)
print('CBOW training complete.')
print('Vocabulary size:', len(cbow_model.wv))

# For downstream tasks we use Skip-Gram as default (better for rare words)
model = sg_model
print()
print('Default model set to: Skip-Gram')
print('Both models available as sg_model and cbow_model')


Training Word2Vec (Skip-Gram) on Quran corpus...
Parameters:
  vector_size = 100  (each word -> 100-dimensional dense vector)
  window      = 5   (5 words on each side as context)
  min_count   = 2   (ignore words appearing < 2 times)
  sg          = 1   (1 = Skip-Gram; 0 would be CBOW)
  epochs      = 50  (passes through the entire corpus)
  workers     = 4   (parallel training threads)

Training complete in 6.0 seconds.

Vocabulary size (words that appear >= 2 times): 3617
Vector dimensions per word: 100

Sample vocabulary words:
['allah', 'unto', 'shall', 'verily', 'thou', 'lord', 'hath', 'say', 'said', 'thee', 'day', 'people', 'may', 'believe', 'earth', 'surely', 'thy', 'sent', 'upon', 'torment']


## Section 4: Exploring Word Similarity

Now we query the trained model to discover what it learned about semantic relationships in the Quran.

In [5]:
# Section 4 — Most similar words to 'mercy'
def print_similar(word, model, topn=10):
    print("Top", topn, "words most similar to:", repr(word))
    print("-" * 45)
    if word not in model.wv:
        print("  [Word not in vocabulary]")
        return
    results = model.wv.most_similar(word, topn=topn)
    for rank, (w, score) in enumerate(results, 1):
        print("  {:2d}. {:20s}  similarity = {:.4f}".format(rank, w, score))
    print()

print_similar('mercy', model)
print_similar('allah', model)
print_similar('prayer', model)
print_similar('punishment', model)

Top 10 words most similar to: 'mercy'
---------------------------------------------
   1. bestower              similarity = 0.4660
   2. heralding             similarity = 0.4515
   3. solicitous            similarity = 0.4435
   4. praiseworthy          similarity = 0.4416
   5. patron                similarity = 0.4375
   6. refrain               similarity = 0.4357
   7. rough                 similarity = 0.4307
   8. perished              similarity = 0.4306
   9. extravagance          similarity = 0.4267
  10. singleth              similarity = 0.4264

Top 10 words most similar to: 'allah'
---------------------------------------------
   1. verily                similarity = 0.6003
   2. perished              similarity = 0.5867
   3. liketh                similarity = 0.5585
   4. tormenteth            similarity = 0.5455
   5. succouring            similarity = 0.5438
   6. unto                  similarity = 0.5418
   7. lord                  similarity = 0.5401
   8. relenting

In [ ]:
# CBOW vs Skip-Gram: Side-by-side most_similar comparison
# This directly answers: which architecture learns better representations?

compare_words = ['mercy', 'prayer', 'punishment', 'believers']
topn = 8

print('CBOW vs SKIP-GRAM: most_similar() comparison')
print('=' * 70)

for word in compare_words:
    cbow_ok = word in cbow_model.wv
    sg_ok   = word in sg_model.wv
    if not cbow_ok and not sg_ok:
        print('Word', repr(word), 'not in vocabulary -- skipping')
        continue
    print('Word:', repr(word))
    print(f'  {"CBOW neighbors":<35} {"Skip-Gram neighbors"}')
    print('  ' + '-' * 65)
    cbow_nbrs = cbow_model.wv.most_similar(word, topn=topn) if cbow_ok else []
    sg_nbrs   = sg_model.wv.most_similar(word, topn=topn)   if sg_ok   else []
    for j in range(topn):
        c_str = f'{cbow_nbrs[j][0]} ({cbow_nbrs[j][1]:.3f})' if j < len(cbow_nbrs) else ''
        s_str = f'{sg_nbrs[j][0]} ({sg_nbrs[j][1]:.3f})'     if j < len(sg_nbrs)   else ''
        print(f'  {c_str:<35} {s_str}')
    print()

print('Key observations:')
print('  CBOW: tends to produce higher similarity scores (tighter clusters)')
print('  Skip-Gram: tends to produce more diverse, semantically rich neighbors')
print('  Skip-Gram generally handles rare words better on small corpora')
print('  CBOW trains faster -- useful when speed matters more than rare word quality')


In [6]:
# Section 4 — Pairwise similarity comparisons
def safe_similarity(word1, word2, model):
    if word1 not in model.wv:
        return None, word1 + ' not in vocab'
    if word2 not in model.wv:
        return None, word2 + ' not in vocab'
    return model.wv.similarity(word1, word2), 'ok'

pairs = [
    ('mercy', 'compassion'),
    ('mercy', 'forgiving'),
    ('mercy', 'fire'),
    ('mercy', 'punishment'),
    ('prayer', 'faith'),
    ('prayer', 'fire'),
    ('allah', 'lord'),
    ('allah', 'punishment')
]

print("Pairwise Cosine Similarities")
print("=" * 50)
print("{:25s}  {:10s}".format("Word Pair", "Similarity"))
print("-" * 50)
for w1, w2 in pairs:
    sim, status = safe_similarity(w1, w2, model)
    if sim is not None:
        bar = '#' * int(sim * 20)
        print("{:12s} vs {:12s}  {:.4f}  {}".format(w1, w2, sim, bar))
    else:
        print("{:12s} vs {:12s}  [{}]".format(w1, w2, status))

print()
print("Interpretation:")
print("  High similarity between 'mercy' and 'compassion' confirms")
print("  these words co-occur in similar Quranic contexts.")
print("  Low similarity between 'mercy' and 'fire' confirms")
print("  they belong to opposite semantic fields.")

Pairwise Cosine Similarities
Word Pair                  Similarity
--------------------------------------------------
mercy        vs compassion    [compassion not in vocab]
mercy        vs forgiving     0.1656  ###
mercy        vs fire          0.2091  ####
mercy        vs punishment    0.1699  ###
prayer       vs faith         0.1688  ###
prayer       vs fire          0.1224  ##
allah        vs lord          0.5401  ##########
allah        vs punishment    0.2757  #####

Interpretation:
  High similarity between 'mercy' and 'compassion' confirms
  these words co-occur in similar Quranic contexts.
  Low similarity between 'mercy' and 'fire' confirms
  they belong to opposite semantic fields.


In [7]:
# Section 4 — Word analogies
print("Word Analogy: king - man + woman = ?")
print("-" * 45)
analogy_words = ['king', 'man', 'woman', 'queen']
in_vocab = [w for w in analogy_words if w in model.wv]
not_in_vocab = [w for w in analogy_words if w not in model.wv]

if not_in_vocab:
    print("Words not in Quran vocabulary:", not_in_vocab)
    print()
    print("This is EXPECTED. The Quran does not discuss royalty in")
    print("the way needed for this analogy to work.")
    print()
    print("The analogy requires that 'king', 'man', 'woman', and 'queen'")
    print("all appear frequently enough (>= min_count=2) in similar contexts.")
    print()
    print("Let us try a Quran-appropriate analogy instead:")
    print()

# Try a domain-appropriate analogy
analogy_tests = [
    (['believers', 'reward'], ['unbelievers'], 'believers - reward + unbelievers = ?'),
    (['mercy', 'compassion'], ['punishment'], 'mercy - compassion + punishment = ?'),
]

for positive, negative, description in analogy_tests:
    pos_in = [w for w in positive if w in model.wv]
    neg_in = [w for w in negative if w in model.wv]
    if len(pos_in) == len(positive) and len(neg_in) == len(negative):
        print("Analogy:", description)
        result = model.wv.most_similar(positive=positive, negative=negative, topn=5)
        for w, s in result:
            print("  {:20s}  {:.4f}".format(w, s))
        print()
    else:
        missing = [w for w in positive + negative if w not in model.wv]
        print("Analogy:", description)
        print("  Skipped — missing from vocab:", missing)
        print()

print("Note: Analogy tasks work best on very large corpora (billions of words).")
print("On ~6000 Quran verses, the geometry is less reliable for analogies")
print("but word similarity is still meaningful and useful.")

Word Analogy: king - man + woman = ?
---------------------------------------------
Words not in Quran vocabulary: ['queen']

This is EXPECTED. The Quran does not discuss royalty in
the way needed for this analogy to work.

The analogy requires that 'king', 'man', 'woman', and 'queen'
all appear frequently enough (>= min_count=2) in similar contexts.

Let us try a Quran-appropriate analogy instead:

Analogy: believers - reward + unbelievers = ?
  greet                 0.4376
  believeth             0.4110
  feareth               0.3933
  lend                  0.3845
  loan                  0.3807

Analogy: mercy - compassion + punishment = ?
  Skipped — missing from vocab: ['compassion']

Note: Analogy tasks work best on very large corpora (billions of words).
On ~6000 Quran verses, the geometry is less reliable for analogies
but word similarity is still meaningful and useful.


## Section 5: Visualising Word Embeddings

Reducing 100 dimensions to 2 using PCA and t-SNE reveals the semantic geometry of the embedding space. Words in the same semantic field should cluster together.

In [8]:
# Section 5 - PCA 2D visualisation with semantic color coding
# Using a larger word set (up to 120 words) for a richer semantic landscape
# Printing explained variance so students know how much info is preserved

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from sklearn.decomposition import PCA

# Semantic categories with representative words
categories = {
    'divine attributes': (['mercy', 'compassion', 'forgiving', 'gracious', 'merciful', 'beneficent'], '#2196F3'),
    'punishment':        (['fire', 'punishment', 'hell', 'torment', 'chastisement', 'wrath'],         '#F44336'),
    'belief':            (['believers', 'faith', 'prayer', 'righteous', 'pious', 'devout'],           '#4CAF50'),
    'prophets':          (['moses', 'abraham', 'jesus', 'noah', 'david', 'solomon'],                  '#FF9800'),
}

# Filter to words actually in vocabulary
cat_words, cat_colors = [], []
for cat_name, (words, color) in categories.items():
    for w in words:
        if w in model.wv:
            cat_words.append(w)
            cat_colors.append(color)

# Add top frequent corpus words to fill out the plot (up to 120 total)
from collections import Counter
all_tokens = [t for verse in tokenized_corpus for t in verse]
freq_words = [w for w, _ in Counter(all_tokens).most_common(200)
              if w in model.wv and w not in cat_words][:max(0, 120 - len(cat_words))]
extra_colors = ['#AAAAAA'] * len(freq_words)

all_plot_words  = cat_words + freq_words
all_plot_colors = cat_colors + extra_colors

word_vectors = np.array([model.wv[w] for w in all_plot_words])

# PCA reduction to 2D
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(word_vectors)

explained = pca.explained_variance_ratio_
print('PCA explained variance:')
print('  PC1:', round(float(explained[0]) * 100, 2), '%')
print('  PC2:', round(float(explained[1]) * 100, 2), '%')
print('  Total:', round(float(explained.sum()) * 100, 2), '% of variance captured in 2D')
print('  (Lower % = more information lost in projection -- expected for 100-dim embeddings)')

# Plot
fig, ax = plt.subplots(figsize=(14, 10))

# Plot background (frequent) words in gray
for i in range(len(cat_words), len(all_plot_words)):
    ax.scatter(pca_result[i, 0], pca_result[i, 1], c='#DDDDDD', s=20, zorder=1)
    ax.annotate(all_plot_words[i], (pca_result[i, 0], pca_result[i, 1]),
                fontsize=6, color='#AAAAAA', zorder=1)

# Plot category words with color
for i in range(len(cat_words)):
    ax.scatter(pca_result[i, 0], pca_result[i, 1], c=all_plot_colors[i], s=120, zorder=3)
    ax.annotate(all_plot_words[i], (pca_result[i, 0], pca_result[i, 1]),
                fontsize=9, fontweight='bold', color='black',
                xytext=(4, 4), textcoords='offset points', zorder=4)

legend_handles = [mpatches.Patch(color=c, label=n.capitalize())
                  for n, (_, c) in categories.items()]
legend_handles.append(mpatches.Patch(color='#AAAAAA', label='Other frequent words'))
ax.legend(handles=legend_handles, loc='upper right', fontsize=10)

ax.set_title(
    'PCA Projection of Word2Vec Embeddings (Quran Corpus)\n'
    'PC1: ' + str(round(float(explained[0])*100,1)) + '% variance  |  '
    'PC2: ' + str(round(float(explained[1])*100,1)) + '% variance  |  '
    'Total: ' + str(round(float(explained.sum())*100,1)) + '% captured',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Principal Component 1 (' + str(round(float(explained[0])*100,1)) + '% variance)')
ax.set_ylabel('Principal Component 2 (' + str(round(float(explained[1])*100,1)) + '% variance)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('word2vec_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('PCA plot saved as word2vec_pca.png')
print('Note: Low explained variance is expected -- 100 dimensions cannot be fully')
print('      captured in 2D. The clusters are still meaningful directionally.')


PCA plot saved as word2vec_pca.png

Observation: Words in the same semantic category should cluster
together. Divine attributes (blue) should be distant from
punishment words (red), reflecting the theological structure of the Quran.


C:\Users\ILI\AppData\Local\Temp\ipykernel_22496\2700504167.py:72: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# Section 5 — t-SNE 2D visualisation
from sklearn.manifold import TSNE

print("Running t-SNE (this may take 30-60 seconds)...")

# Use all category words + some extras for a richer t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=min(30, len(all_plot_words) - 1),
    max_iter=1000,
    random_state=42,
    learning_rate='auto',
    init='pca'
)
tsne_result = tsne.fit_transform(word_vectors)

fig, ax = plt.subplots(figsize=(14, 10))

for i in range(len(words_to_plot), len(all_plot_words)):
    ax.scatter(tsne_result[i, 0], tsne_result[i, 1], c='#DDDDDD', s=20, zorder=1)
    ax.annotate(all_plot_words[i], (tsne_result[i, 0], tsne_result[i, 1]),
                fontsize=6, color='#AAAAAA', zorder=1)

for i in range(len(words_to_plot)):
    ax.scatter(tsne_result[i, 0], tsne_result[i, 1], c=all_colors[i], s=120, zorder=3)
    ax.annotate(words_to_plot[i], (tsne_result[i, 0], tsne_result[i, 1]),
                fontsize=9, fontweight='bold', color='black',
                xytext=(4, 4), textcoords='offset points', zorder=4)

legend_handles = []
for cat_name, (_, color) in categories.items():
    legend_handles.append(mpatches.Patch(color=color, label=cat_name.capitalize()))
ax.legend(handles=legend_handles, loc='upper right', fontsize=10)

ax.set_title('t-SNE Projection of Word2Vec Embeddings (Quran Corpus)\nt-SNE preserves local neighbourhood structure better than PCA', fontsize=13)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('word2vec_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print("t-SNE plot saved as word2vec_tsne.png")
print()
print("Key difference from PCA:")
print("  PCA is a linear projection — preserves global variance.")
print("  t-SNE is non-linear — prioritises keeping similar words close.")
print("  t-SNE clusters tend to be tighter and more interpretable.")
print("  However, t-SNE distances between clusters are NOT meaningful.")

Running t-SNE (this may take 30-60 seconds)...
t-SNE plot saved as word2vec_tsne.png

Key difference from PCA:
  PCA is a linear projection — preserves global variance.
  t-SNE is non-linear — prioritises keeping similar words close.
  t-SNE clusters tend to be tighter and more interpretable.
  However, t-SNE distances between clusters are NOT meaningful.


C:\Users\ILI\AppData\Local\Temp\ipykernel_22496\1258857216.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 6: Document Embeddings for Search

### The Problem: Word Vectors, Not Document Vectors

Word2Vec gives us a vector for each *word*, not each *verse* or *document*. To do semantic search over the Quran, we need to represent each verse as a single vector.

Three standard strategies exist:

---

### Strategy 1: Average Word Vectors (Simple Mean Pooling)

```
verse = "He is the most merciful and forgiving"

word vectors:
  merciful  -> [0.32, -0.14, 0.87, ...]
  forgiving -> [0.28, -0.11, 0.79, ...]
  ... (stop words excluded)

verse_vector = MEAN of all word vectors  -> [0.30, -0.12, 0.83, ...]
```

Simple, fast, and surprisingly effective. The main drawback: all words are treated equally — common words dilute the meaning.

---

### Strategy 2: TF-IDF Weighted Average

```
verse_vector = SUM(tfidf(word) * word_vector(word))  /  SUM(tfidf(word))
```

Rare, informative words get higher weight. This is better for retrieval tasks because content words dominate.

---

### Strategy 3: Max Pooling (Mentioned for Completeness)

```
verse_vector[i] = MAX over all words of word_vector[i]
```

Takes the maximum value at each dimension across all word vectors. Captures the most prominent feature in each dimension. Less common in practice but sometimes used in CNNs for text.

In [10]:
# Section 6 — Build verse embedding matrices (Strategy 1 and 2)
from sklearn.feature_extraction.text import TfidfVectorizer

# --- Strategy 1: Simple mean pooling ---
def verse_to_vector_mean(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

print("Building mean-pooled verse embeddings...")
verse_texts = df['daryabadi'].dropna().tolist()
verse_tokens = [tokenize_verse(v) for v in verse_texts]

verse_vectors_mean = np.array([verse_to_vector_mean(t, model) for t in verse_tokens])
print("Mean-pooled matrix shape:", verse_vectors_mean.shape)

# --- Strategy 2: TF-IDF weighted mean ---
print()
print("Building TF-IDF vocabulary for weighting...")
cleaned_texts = [' '.join(t) for t in verse_tokens]
tfidf_vec = TfidfVectorizer(max_features=10000)
tfidf_matrix = tfidf_vec.fit_transform(cleaned_texts)
tfidf_feature_names = tfidf_vec.get_feature_names_out()
word_to_tfidf_idx = {w: i for i, w in enumerate(tfidf_feature_names)}

def verse_to_vector_tfidf(tokens, model, tfidf_row, word_to_tfidf_idx):
    vecs = []
    weights = []
    for w in tokens:
        if w in model.wv and w in word_to_tfidf_idx:
            idx = word_to_tfidf_idx[w]
            weight = tfidf_row[0, idx]
            vecs.append(model.wv[w])
            weights.append(weight)
    if not vecs:
        return np.zeros(model.vector_size)
    vecs = np.array(vecs)
    weights = np.array(weights)
    total_weight = weights.sum()
    if total_weight == 0:
        return np.mean(vecs, axis=0)
    return np.average(vecs, axis=0, weights=weights)

print("Building TF-IDF weighted verse embeddings...")
verse_vectors_tfidf = np.array([
    verse_to_vector_tfidf(verse_tokens[i], model, tfidf_matrix[i], word_to_tfidf_idx)
    for i in range(len(verse_tokens))
])
print("TF-IDF weighted matrix shape:", verse_vectors_tfidf.shape)
print()
print("Both embedding matrices are ready for semantic search.")
print("Each row is a 100-dimensional dense representation of one Quranic verse.")

Building mean-pooled verse embeddings...
Mean-pooled matrix shape: (6236, 100)

Building TF-IDF vocabulary for weighting...
Building TF-IDF weighted verse embeddings...
TF-IDF weighted matrix shape: (6236, 100)

Both embedding matrices are ready for semantic search.
Each row is a 100-dimensional dense representation of one Quranic verse.


## Section 7: Word2Vec Semantic Search Engine

We now have everything we need to build a semantic search engine. Unlike NB01-NB03 (which required exact word matches), this engine understands *meaning*.

In [11]:
# Section 7 — Semantic search function using Word2Vec embeddings
from sklearn.metrics.pairwise import cosine_similarity

def semantic_search_w2v(query, verse_vectors, verse_texts, model, top_k=5):
    query_tokens = tokenize_verse(query)
    query_vec = verse_to_vector_mean(query_tokens, model)
    if np.linalg.norm(query_vec) == 0:
        print("Query vector is zero — no query words found in vocabulary.")
        return []
    sims = cosine_similarity([query_vec], verse_vectors)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    results = [(i, sims[i], verse_texts[i]) for i in top_indices]
    return results

def display_search_results(query, results):
    print("Query:", repr(query))
    print("=" * 70)
    for rank, (idx, score, text) in enumerate(results, 1):
        snippet = text[:120] + '...' if len(text) > 120 else text
        print("Rank {}  [score={:.4f}]  Verse #{}".format(rank, score, idx))
        print("  ", snippet)
        print()

print("=" * 70)
print("QUERY 1: 'kindness' — THE NB01 FAILURE CASE")
print("In NB01, searching 'kindness' returned 0 results (exact match failed).")
print("Now watch Word2Vec retrieve mercy/compassion verses semantically.")
print("=" * 70)
results_kindness = semantic_search_w2v('kindness', verse_vectors_mean, verse_texts, model, top_k=5)
display_search_results('kindness', results_kindness)

QUERY 1: 'kindness' — THE NB01 FAILURE CASE
In NB01, searching 'kindness' returned 0 results (exact match failed).
Now watch Word2Vec retrieve mercy/compassion verses semantically.
Query: 'kindness'
Rank 1  [score=0.8400]  Verse #4960
   Shall the recompense of kindness be aught save kindness?

Rank 2  [score=0.6708]  Verse #2051
   And thy Lord hath decreed that ye shall Worship none but Him, and unto parents shew kindness; and if either of them or b...

Rank 3  [score=0.6438]  Verse #3347
   And We have enjoined on man kindness unto parents. But if the twain strive to make thee associate with Me that of which ...

Rank 4  [score=0.6411]  Verse #4524
   And We have enjoined upon man kindness Unto the parents: with hardship his mother beareth him and with hardship she brin...

Rank 5  [score=0.6281]  Verse #528
   And worship Allah and associate not aught with Him; and unto parents show kindness, and also unto kindred and orphans an...



In [12]:
# Section 7 — More semantic search queries
print("=" * 70)
print("QUERY 2: 'divine compassion' — multi-word semantic query")
print("=" * 70)
results_compassion = semantic_search_w2v('divine compassion', verse_vectors_mean, verse_texts, model, top_k=5)
display_search_results('divine compassion', results_compassion)

print("=" * 70)
print("QUERY 3: 'punishment of the wicked'")
print("=" * 70)
results_punishment = semantic_search_w2v('punishment of the wicked', verse_vectors_mean, verse_texts, model, top_k=5)
display_search_results('punishment of the wicked', results_punishment)

QUERY 2: 'divine compassion' — multi-word semantic query
Query vector is zero — no query words found in vocabulary.
Query: 'divine compassion'
QUERY 3: 'punishment of the wicked'
Query: 'punishment of the wicked'
Rank 1  [score=0.7064]  Verse #3562
   O Ye wives of the Prophet! whosoever of you shall commit a manifest Indecency, doubled for her would be the punishment t...

Rank 2  [score=0.7008]  Verse #706
   As for the man-thief and the woman-thief, cut off their hands as a meed for that which they have earned; an exemplary pu...

Rank 3  [score=0.6813]  Verse #2813
   verily those who accuse chaste. unknowing, - believing women, shall be cursed in the world and the Hereafter; and for th...

Rank 4  [score=0.6698]  Verse #5736
   Wherefore Allah laid hold of him with the punishment of the Hereafter and of the present.

Rank 5  [score=0.6669]  Verse #4066
   Is he who is devout, in the watches of the night prostrating himself and standing, bewaring of the Hereafter and hoping ...



## Section 8: Word2Vec vs TF-IDF Comparison

Let us formally compare the two approaches on the query that exposed BoW's fundamental limitation: **"kindness"**.

In [13]:
# Section 8 — TF-IDF search for comparison
from sklearn.metrics.pairwise import cosine_similarity as cos_sim_fn

def tfidf_search(query, tfidf_vec, tfidf_matrix, verse_texts, top_k=5):
    query_tfidf = tfidf_vec.transform([query])
    sims = cos_sim_fn(query_tfidf, tfidf_matrix)[0]
    top_indices = sims.argsort()[::-1][:top_k]
    results = [(i, sims[i], verse_texts[i]) for i in top_indices]
    return results

query = 'kindness'

print("Query:", repr(query))
print()
print("--- TF-IDF Results ---")
tfidf_results = tfidf_search(query, tfidf_vec, tfidf_matrix, verse_texts, top_k=5)
for rank, (idx, score, text) in enumerate(tfidf_results, 1):
    snippet = text[:100] + '...' if len(text) > 100 else text
    print("  Rank {}  [score={:.4f}]:  {}".format(rank, score, snippet))

print()
print("--- Word2Vec Semantic Results ---")
w2v_results = semantic_search_w2v(query, verse_vectors_mean, verse_texts, model, top_k=5)
for rank, (idx, score, text) in enumerate(w2v_results, 1):
    snippet = text[:100] + '...' if len(text) > 100 else text
    print("  Rank {}  [score={:.4f}]:  {}".format(rank, score, snippet))

Query: 'kindness'

--- TF-IDF Results ---
  Rank 1  [score=0.8389]:  Shall the recompense of kindness be aught save kindness?
  Rank 2  [score=0.3179]:  How then, when some ill befalleth them because of that which their hands have sent forth and then th...
  Rank 3  [score=0.3089]:  And We have enjoined on man kindness unto parents. But if the twain strive to make thee associate wi...
  Rank 4  [score=0.2601]:  No good is there in much of their whispers except in his who commandeth charity or kindness or recon...
  Rank 5  [score=0.2475]:  And thy Lord hath decreed that ye shall Worship none but Him, and unto parents shew kindness; and if...

--- Word2Vec Semantic Results ---
  Rank 1  [score=0.8400]:  Shall the recompense of kindness be aught save kindness?
  Rank 2  [score=0.6708]:  And thy Lord hath decreed that ye shall Worship none but Him, and unto parents shew kindness; and if...
  Rank 3  [score=0.6438]:  And We have enjoined on man kindness unto parents. But if the twain striv

In [14]:
# Section 8 — Bar chart comparison of similarity scores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# TF-IDF scores
tfidf_scores = [s for _, s, _ in tfidf_results]
tfidf_labels = ['Rank {}'.format(i+1) for i in range(len(tfidf_results))]
bars1 = ax1.bar(tfidf_labels, tfidf_scores, color='#FF7043', edgecolor='black')
ax1.set_title('TF-IDF Results for "kindness"\n(exact match only)', fontsize=12)
ax1.set_ylabel('Cosine Similarity')
ax1.set_ylim(0, 1)
for bar, score in zip(bars1, tfidf_scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             '{:.4f}'.format(score), ha='center', va='bottom', fontsize=9)

# Word2Vec scores
w2v_scores = [s for _, s, _ in w2v_results]
w2v_labels = ['Rank {}'.format(i+1) for i in range(len(w2v_results))]
bars2 = ax2.bar(w2v_labels, w2v_scores, color='#42A5F5', edgecolor='black')
ax2.set_title('Word2Vec Results for "kindness"\n(semantic similarity)', fontsize=12)
ax2.set_ylabel('Cosine Similarity')
ax2.set_ylim(0, 1)
for bar, score in zip(bars2, w2v_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             '{:.4f}'.format(score), ha='center', va='bottom', fontsize=9)

plt.suptitle('TF-IDF vs Word2Vec: Query = "kindness"', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('tfidf_vs_w2v.png', dpi=150, bbox_inches='tight')
plt.show()
print("Comparison chart saved as tfidf_vs_w2v.png")
print()
print("Key insight:")
print("  TF-IDF scores near 0 = 'kindness' literally does not appear in top results.")
print("  Word2Vec scores higher = model knows 'kindness' is semantically close")
print("  to 'mercy', 'compassion', 'forgiveness' because those words share context.")
print()
print("WHY Word2Vec succeeds:")
print("  'kindness' appears in similar sentence positions as 'mercy'.")
print("  The Skip-Gram model assigns them nearby vectors in 100D space.")
print("  Cosine similarity in that space finds mercy/compassion verses.")

Comparison chart saved as tfidf_vs_w2v.png

Key insight:
  TF-IDF scores near 0 = 'kindness' literally does not appear in top results.
  Word2Vec scores higher = model knows 'kindness' is semantically close
  to 'mercy', 'compassion', 'forgiveness' because those words share context.

WHY Word2Vec succeeds:
  'kindness' appears in similar sentence positions as 'mercy'.
  The Skip-Gram model assigns them nearby vectors in 100D space.
  Cosine similarity in that space finds mercy/compassion verses.


C:\Users\ILI\AppData\Local\Temp\ipykernel_22496\1965520716.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Section 9: Limitations of Word2Vec

Despite its power, Word2Vec has fundamental limitations that motivated subsequent research. Understanding these prepares us for FastText (NB05) and Sentence Transformers (NB06).

---

### Limitation 1: One Vector Per Word — The Polysemy Problem

Word2Vec assigns a single vector to each word token, regardless of context. But many words have multiple meanings:

```
"bank"  could mean:
  (a) river bank   -- geological feature
  (b) financial bank -- institution

Word2Vec gives BOTH senses the SAME vector:
  bank -> [0.21, -0.33, 0.71, ...]   (one fixed vector)

The model averages over all contexts it saw "bank" in.
If the corpus has both uses, the vector is somewhere in between
and accurately represents neither.
```

This is the **polysemy problem**. BERT (NB07) solves this with contextual embeddings.

---

### Limitation 2: No Sentence-Level Understanding

Averaging word vectors loses grammatical structure:

```
"The merciful punish the believers"  (meaning: merciful beings punish)
"The believers punish the merciful"  (meaning: believers punish merciful beings)

Both sentences have the SAME average word vector (same words, different order).
```

Word order, syntax, and long-range dependencies are invisible to a mean-pooled representation.

---

### Limitation 3: Requires Large Corpus

The quality of embeddings depends on the quantity and diversity of training data:

- 6,236 Quran verses: small — some words appear rarely, embeddings are noisy
- Wikipedia (3B words): much better general-purpose embeddings
- Common Crawl (800B words): state-of-the-art quality

Our Quran-trained model captures religious semantics well but may fail on general-purpose analogies.

---

### Limitation 4: OOV (Out-of-Vocabulary) Problem

If a word was not in the training corpus (or appeared fewer than min_count=2 times), it has **no vector**. This is fatal for:

- Named entities not in the training data
- Technical terms, neologisms, rare words
- Any query containing an unknown word

FastText (NB05) solves OOV by representing words as sums of character n-grams — even unknown words have a vector.

---

### Limitation-to-Next-Step Table

```
+------------------------------+----------------------------+------------------+
| Limitation                   | Consequence                | Fixed In         |
+------------------------------+----------------------------+------------------+
| One vector per word          | Cannot handle polysemy     | BERT (NB07)      |
| OOV words crash              | Missing rare terms         | FastText (NB05)  |
| No morphology awareness      | run/running treated diff.  | FastText (NB05)  |
| Averaging loses word order   | Same vector, diff. meaning | SBERT (NB06)     |
| Needs large training corpus  | Noisy on 6K verses         | Pretrained models|
+------------------------------+----------------------------+------------------+
```

In [15]:
# Section 9 — Demo: OOV problem and polysemy

# Demo 1: OOV word
print("=== Demo 1: Out-of-Vocabulary (OOV) Problem ===")
oov_candidates = ['cryptocurrency', 'smartphone', 'algorithm', 'satellite']

for word in oov_candidates:
    if word not in model.wv:
        print("model.wv['{}'] --> KeyError: word not in vocabulary".format(word))
        break

print()
print("Trying to access OOV word:")
test_word = next((w for w in oov_candidates if w not in model.wv), None)
if test_word:
    try:
        vec = model.wv[test_word]
    except KeyError as e:
        print("  KeyError:", e)
        print("  The word '{}' was never seen during training.".format(test_word))
        print("  Word2Vec has NO representation for it.")
else:
    print("  All test OOV words happened to be in vocab. Quran is large!")
    print("  Try model.wv['fluorocarbon'] for a guaranteed OOV.")

print()
print("=== Demo 2: One Vector Per Word (Polysemy) ===")

# Check if 'right' is in vocab (has multiple meanings)
for ambiguous_word in ['right', 'lord', 'good', 'light', 'sign']:
    if ambiguous_word in model.wv:
        vec = model.wv[ambiguous_word]
        print("Word: '{}' (potentially ambiguous)".format(ambiguous_word))
        print("  Has ONE vector of shape:", vec.shape)
        print("  All meanings (moral right, direction right, etc.) merged.")
        print("  Top 5 neighbours:", [w for w, _ in model.wv.most_similar(ambiguous_word, topn=5)])
        print()
        break

print("Conclusion: Word2Vec cannot distinguish 'right' (correct) from")
print("'right' (direction) — they share one fixed vector averaged over all contexts.")
print("BERT (NB07) generates a DIFFERENT vector per occurrence based on context.")

=== Demo 1: Out-of-Vocabulary (OOV) Problem ===
model.wv['cryptocurrency'] --> KeyError: word not in vocabulary

Trying to access OOV word:
  KeyError: "Key 'cryptocurrency' not present"
  The word 'cryptocurrency' was never seen during training.
  Word2Vec has NO representation for it.

=== Demo 2: One Vector Per Word (Polysemy) ===
Word: 'right' (potentially ambiguous)
  Has ONE vector of shape: (100,)
  All meanings (moral right, direction right, etc.) merged.
  Top 5 neighbours: ['minded', 'blameworthy', 'miserable', 'hand', 'beggar']

Conclusion: Word2Vec cannot distinguish 'right' (correct) from
'right' (direction) — they share one fixed vector averaged over all contexts.
BERT (NB07) generates a DIFFERENT vector per occurrence based on context.


## Section 10: Student Reflection Questions

These questions are designed to test your conceptual understanding. Write your answers before running any further code.

---

**Question 1:**  
Why does Word2Vec place "mercy" and "compassion" close together in vector space? Trace the reasoning from the raw text corpus all the way to the final vector positions.

*Think about: What does the Skip-Gram model learn to predict? What happens to weights when two words appear in similar contexts? How does this affect vector distance?*

---

**Question 2:**  
What is the distributional hypothesis (Firth, 1957) and why does it provide the theoretical foundation for learning semantic embeddings from text alone — without any manually labelled data?

*Think about: What does "company it keeps" mean formally? Why is co-occurrence a proxy for meaning? When would this hypothesis fail?*

---

**Question 3:**  
Why does averaging word vectors lose information compared to a true sentence embedding? Give a concrete example using two sentences that would have identical averaged vectors.

*Think about: What information is destroyed when you sum/average? What linguistic phenomena require order to be preserved?*

---

**Question 4:**  
The Quran corpus contains approximately 6,000 verses — a very small corpus by NLP standards. How would you expect the quality of embeddings to improve if you trained Word2Vec on 1 million sentences from Islamic scholarly texts? What specifically would improve, and why?

*Think about: Frequency of rare theological terms, quality of analogies, cluster tightness, similarity scores.*

---

**Question 5:**  
What happens when a user queries the semantic search engine with a word not in the training vocabulary (e.g., "cryptocurrency")? Describe the failure mode step by step and explain how FastText (NB05) solves this problem at the architectural level.

*Think about: What does tokenize_verse return? What does verse_to_vector_mean return when no tokens match? What would the cosine similarity be against real verse vectors?*

## Section 11: Summary and What's Next

### What We Covered in NB04

```
+-----------------------------------------------+-------------------------------------+
| Topic                                         | Key Takeaway                        |
+-----------------------------------------------+-------------------------------------+
| Distributional hypothesis                     | Words known by company they keep    |
| Sparse vs dense vectors                       | Dense = semantically structured     |
| CBOW architecture                             | Predict center from context         |
| Skip-Gram architecture                        | Predict context from center (sg=1)  |
| Training Word2Vec on Quran corpus             | 100D vectors from ~6K verses        |
| Word similarity exploration                   | mercy ~ compassion; mercy !~ fire   |
| PCA + t-SNE visualisation                    | Semantic clusters visible in 2D     |
| Mean pooling for document vectors             | Average word vecs = verse vector    |
| TF-IDF weighted pooling                       | Rare words get more weight          |
| Semantic search engine                        | 'kindness' retrieves mercy verses   |
| Word2Vec vs TF-IDF comparison                 | W2V wins on semantic queries        |
| Polysemy limitation                           | One vector per word, context-blind  |
| OOV limitation                               | Unknown words crash at query time   |
+-----------------------------------------------+-------------------------------------+
```

---

### The NB01 Problem — Solved

In NB01, searching for "kindness" returned zero results. The word does not appear verbatim in the Quran translation.

In NB04, the same query retrieves the top mercy and compassion verses — because our semantic search engine understands that kindness, mercy, and compassion occupy the same region of embedding space.

This is the practical payoff of the theoretical investment in Word2Vec.

---

### Preview: NB05 — FastText

FastText (Joulin et al., Facebook, 2017) extends Word2Vec with two key innovations:

**1. Subword representations**  
Instead of one vector per word, FastText represents each word as the *sum of its character n-gram vectors*:
```
"merciful" = "<me" + "mer" + "erc" + "rci" + "cif" + "ifu" + "ful" + "ul>" + ...
```
This means:
- Morphologically related words share subword vectors automatically
- OOV words like "unmerciful" get a vector from their character n-grams

**2. OOV Handling**  
Any word — even one never seen in training — gets a vector from its character composition. This is critical for Arabic names, transliterations, and rare theological terms.

---

### Limitation-to-Next-Step Map

```
NB04 Word2Vec Limitation          -->  Solution in
-------------------------------        -------------------------
OOV words have no vector          -->  NB05: FastText (char n-grams)
Morphology ignored (run/running)  -->  NB05: FastText subword model
One vector ignores polysemy       -->  NB07: BERT contextual embeddings
Averaging loses sentence order    -->  NB06: SBERT sentence embeddings
Needs training from scratch       -->  NB06/07: Pretrained models
```

---

*End of NB04 — Proceed to NB05: FastText and Subword Embeddings*